# VAE latent-space & β experiments

Companion to `VAE.ipynb`, restructured for **systematic experiments** on how the
KL weight **β** shapes the latent space.

What's different from the base notebook:
- **Colab-ready** setup cell (clone repo + install + mount Drive).
- Training is wrapped in `train_vae(...)` so it can be called once per β.
- A **β sweep** trains several models and overlays their curves.
- Latent diagnostics (`explore_latent`, `latent_physical_alignment`) take a
  `model` argument so any swept model can be inspected.

Run the cells top to bottom. Set `BETAS` in the sweep cell to whatever grid you
want to explore.

In [1]:
# ── Colab setup ────────────────────────────────────────────────────────────
# Detects Colab, clones the repo, installs it, and mounts Drive for data/ckpts.
# On a local machine this whole cell is a no-op.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/choROPeNt/improved-diffusion.git"
    REPO_DIR = "/content/improved-diffusion"
    BRANCH   = "dev"

    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
    # editable install so `import improved_diffusion` works
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "torchinfo", "h5py", "scipy", "tqdm"], check=True)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    from google.colab import drive
    drive.mount("/content/drive")

    # ── EDIT ME: where your data + checkpoints live on Drive ──────────────
    DRIVE_ROOT = "/content/drive/MyDrive/improved-diffusion"
    DATA_PATH  = os.path.join(DRIVE_ROOT, "data", "patches_vae.h5")
    CKPT_DIR   = os.path.join(DRIVE_ROOT, "experiments", "vae_beta")
else:
    # local paths (mirrors VAE.ipynb)
    DATA_PATH = "../data/patches_vae.h5"
    CKPT_DIR  = "../experiments/vae_beta"

os.makedirs(CKPT_DIR, exist_ok=True)
print(f"IN_COLAB  = {IN_COLAB}")
print(f"DATA_PATH = {DATA_PATH}")
print(f"CKPT_DIR  = {CKPT_DIR}")

IN_COLAB  = False
DATA_PATH = ../data/patches_vae.h5
CKPT_DIR  = ../experiments/vae_beta


In [2]:
from pathlib import Path

import numpy as np
import h5py
import json
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader, Subset
from torchinfo import summary
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from improved_diffusion.models.vae import AbstractVAE

In [3]:
def vae_loss(x_hat, x, mu, logvar, beta=1.0, recon="mse"):
    if recon == "bce":
        rec = F.binary_cross_entropy_with_logits(x_hat, x, reduction="mean")
    else:
        rec = F.mse_loss(x_hat, x, reduction="mean")
    # KL(q(z|x) || N(0,I))
    kl = -0.5 * torch.mean(1.0 + logvar - mu.pow(2) - logvar.exp())
    return rec + beta * kl, rec, kl

In [4]:
DEVICE = torch.device(
    "mps"  if torch.backends.mps.is_available()  else
    "cuda" if torch.cuda.is_available()           else
    "cpu"
)
print(f"device: {DEVICE}")

device: mps


In [5]:
from torch.utils.data import Dataset, DataLoader
import numpy as np
import h5py
import json

class PatchDataset(Dataset):
    """
    HDF5-backed dataset with paired binary and grayscale patches.

    Returns (binary [1,H,W] float32 ∈ {0,1},
             gray   [1,H,W] float32 ∈ [0,1],
             phi    float32,
             source str)   ← originating filename, e.g. '245_00.h5'

    File handle opened lazily per worker — num_workers>0 safe.
    """

    def __init__(self, h5_path, class_filter=None, source_filter=None, transform=None):
        self.h5_path   = str(h5_path)
        self.transform = transform
        self._file     = None  # opened lazily per worker

        with h5py.File(self.h5_path, "r") as f:
            self.class_names  = json.loads(f.attrs["class_names"])
            self.source_files = json.loads(f.attrs["source_files"])
            class_ids = f["class_id"][:]
            sources   = f["source"][:].astype(str)  # str array

        mask = np.ones(len(class_ids), dtype=bool)
        if class_filter is not None:
            keep = {self.class_names.index(c) for c in class_filter}
            mask &= np.isin(class_ids, list(keep))
        if source_filter is not None:
            mask &= np.isin(sources, source_filter)
        self._indices = np.where(mask)[0]

    def __len__(self):
        return len(self._indices)

    def __getitem__(self, i):
        if self._file is None:
            self._file = h5py.File(self.h5_path, "r")
        idx = int(self._indices[i])

        binary = torch.from_numpy(
            self._file["patches"][idx].astype(np.float32)
        ).unsqueeze(0)                                           # [1, H, W] ∈ {0,1}
        gray = torch.from_numpy(
            self._file["images"][idx].astype(np.float32) / 255.0
        ).unsqueeze(0)                                           # [1, H, W] ∈ [0,1]
        phi    = torch.tensor(float(self._file["phi"][idx]))
        s      = self._file["source"][idx]
        source = s.decode() if isinstance(s, bytes) else str(s)  # h5py 2/3 compat

        if self.transform is not None:
            binary = self.transform(binary)
            gray   = self.transform(gray)
        return binary, gray, phi, source

In [6]:
# ── dataset + stratified train/val split ───────────────────────────────────
# Hold out a DISJOINT validation subset. Reconstruction loss is only a fair
# metric on data the model did not train on. We STRATIFY by φ (fiber volume
# fraction) so train and val share the same φ distribution — a plain random
# split can hand val a skewed (e.g. too-high) φ range by chance.
VAL_FRAC   = 1/4
N_BINS     = 10        # φ quantile bins to stratify over
BATCH_SIZE = 4

ds = PatchDataset(DATA_PATH)
print(f"dataset: {len(ds)} patches")

# φ for every sample (read straight from the h5 file, ordered like ds)
with h5py.File(DATA_PATH, "r") as _f:
    phi_all = _f["phi"][:][ds._indices].astype(np.float64)

# equal-count quantile bins, then hold out VAL_FRAC within each bin
_edges  = np.quantile(phi_all, np.linspace(0, 1, N_BINS + 1))
_bin_id = np.clip(np.digitize(phi_all, _edges[1:-1]), 0, N_BINS - 1)

_rng = np.random.default_rng(0)        # deterministic split
val_mask = np.zeros(len(phi_all), dtype=bool)
for b in range(N_BINS):
    idx_b = np.where(_bin_id == b)[0]
    _rng.shuffle(idx_b)
    n_val_b = int(round(len(idx_b) * VAL_FRAC))
    val_mask[idx_b[:n_val_b]] = True

val_idx   = np.where(val_mask)[0].tolist()
train_idx = np.where(~val_mask)[0].tolist()
train_subset = Subset(ds, train_idx)
val_subset   = Subset(ds, val_idx)
assert not (set(train_idx) & set(val_idx)), "train/val overlap!"

phi_tr, phi_va = phi_all[train_idx], phi_all[val_idx]
print(f"train: {len(train_subset)}   val: {len(val_subset)}  "
      f"({len(train_subset)/len(ds):.0%} / {len(val_subset)/len(ds):.0%}, disjoint)")
print(f"φ  train: mean={phi_tr.mean():.4f} std={phi_tr.std():.4f} "
      f"range=[{phi_tr.min():.3f}, {phi_tr.max():.3f}]")
print(f"φ  val:   mean={phi_va.mean():.4f} std={phi_va.std():.4f} "
      f"range=[{phi_va.min():.3f}, {phi_va.max():.3f}]")

def _collate_downscale(batch):
    binary_list, gray_list, phi_list, _ = zip(*batch)
    x = torch.stack(binary_list)                     # [B, 1, 512, 512] ∈ {0,1}
    x = F.avg_pool2d(x, kernel_size=2, stride=2)    # [B, 1, 256, 256]
    x = (x > 0.5).float() * 2.0 - 1.0              # re-threshold → {-1, 1}
    phi = torch.stack(phi_list)                      # [B] fiber volume fraction
    return x, phi

train_loader = DataLoader(
    train_subset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2 if IN_COLAB else 0,
    collate_fn=_collate_downscale, drop_last=False,
)
val_loader = DataLoader(
    val_subset, batch_size=BATCH_SIZE, shuffle=False,   # no shuffle → stable metric
    num_workers=2 if IN_COLAB else 0,
    collate_fn=_collate_downscale, drop_last=False,
)

_xb, _phi = next(iter(train_loader))
print(f"x:   {tuple(_xb.shape)}   values: {_xb.unique().tolist()}")
print(f"phi: {tuple(_phi.shape)}  range:  [{_phi.min():.3f}, {_phi.max():.3f}]")

def _cycle(dl):
    while True:
        yield from dl

dataset: 2250 patches
train: 1690   val: 560  (75% / 25%, disjoint)
φ  train: mean=0.7160 std=0.0434 range=[0.362, 0.796]
φ  val:   mean=0.7145 std=0.0500 range=[0.348, 0.794]
x:   (4, 1, 256, 256)   values: [-1.0, 1.0]
phi: (4,)  range:  [0.670, 0.715]


In [ ]:
def build_vae(latent_channels=4, base_channels=32, channel_mult=(1, 2, 4, 4),
              attn_ds=(4, 8), attn_heads=1, **kwargs):
    """Fresh model — same architecture as VAE.ipynb, but architecture knobs are
    now arguments so we can sweep them. Call per experiment; seed beforehand to
    fix the random init."""
    return AbstractVAE(
        in_channels=1,
        latent_channels=latent_channels,
        base_channels=base_channels,
        channel_mult=channel_mult,
        dims=2,
        attn_ds=attn_ds,
        attn_heads=attn_heads,
        spatial_latent=True,
        **kwargs,
    ).to(DEVICE)

# sanity check
_vae = build_vae()
_dummy = torch.zeros(2, 1, 256, 256, device=DEVICE)
with torch.no_grad():
    _out, _mu, _lv, _z = _vae(_dummy)
print(f"latent:  {tuple(_z.shape)}")
print(f"params:  {sum(p.numel() for p in _vae.parameters()):,}")
del _vae

In [ ]:
@torch.no_grad()
def eval_recon(model, loader):
    """Mean reconstruction MSE over a loader, decoding the latent *mean* μ
    (deterministic — no sampling noise, so the metric is stable across calls).
    This is the fair, comparable number for picking β / latent_channels."""
    was_training = model.training
    model.eval()
    total, n = 0.0, 0
    for x, _phi in loader:
        x = x.to(DEVICE)
        mu, logvar = model.encode(x)
        x_hat = model.decode(mu)
        total += F.mse_loss(x_hat, x, reduction="mean").item() * x.shape[0]
        n     += x.shape[0]
    if was_training:
        model.train()
    return total / max(n, 1)


def train_vae(
    beta_max,
    beta_warmup_steps=1000,
    total_steps=3000,
    lr=1e-3,
    grad_clip=1.0,
    log_interval=50,
    seed=0,
    progress=True,
    desc=None,
    model_kwargs=None,
    val_loader=None,
):
    """Train one VAE at a given β schedule. Returns (model, history).

    Seeding before build_vae() makes runs directly comparable (same init + same
    data order). `model_kwargs` is forwarded to build_vae() so architecture
    sweeps (e.g. latent_channels) reuse this exact loop. When `val_loader` is
    given, held-out reconstruction MSE is logged to history["val_recon"] — that
    is the metric to compare across runs (train recon just measures memorization)."""
    torch.manual_seed(seed)
    np.random.seed(seed)

    vae = build_vae(**(model_kwargs or {}))
    opt = AdamW(vae.parameters(), lr=lr, weight_decay=1e-5)
    data_iter = _cycle(train_loader)

    def beta_schedule(step):
        if beta_warmup_steps <= 0:
            return beta_max
        return beta_max * min(1.0, step / beta_warmup_steps)

    history = {"step": [], "loss": [], "recon": [], "val_recon": [],
               "kl": [], "beta": [], "sigma_mean": []}
    vae.train()
    desc = desc if desc is not None else f"β={beta_max:.0e}"
    iterator = tqdm(range(total_steps), leave=False, desc=desc) if progress else range(total_steps)

    for step in iterator:
        x, _phi = next(data_iter)
        x = x.to(DEVICE)
        beta = beta_schedule(step)

        opt.zero_grad(set_to_none=True)
        x_hat, mu, logvar, z = vae(x)
        loss, rec, kl = vae_loss(x_hat, x, mu, logvar, beta=beta)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(vae.parameters(), grad_clip)
        opt.step()

        if step % log_interval == 0:
            sigma_mean = logvar.mul(0.5).exp().mean().item()
            val_rec = eval_recon(vae, val_loader) if val_loader is not None else float("nan")
            history["step"].append(step)
            history["loss"].append(loss.item())
            history["recon"].append(rec.item())          # train recon (this batch)
            history["val_recon"].append(val_rec)         # held-out recon
            history["kl"].append(kl.item())
            history["beta"].append(beta)
            history["sigma_mean"].append(sigma_mean)
            if progress:
                iterator.set_postfix(rec=f"{rec.item():.4f}", val=f"{val_rec:.4f}",
                                     kl=f"{kl.item():.3f}", sig=f"{sigma_mean:.3f}",
                                     beta=f"{beta:.1e}")

    return vae, history

## β sweep

Each β trains a separate model from the **same seed** (identical init + data
order), so differences in the curves are attributable to β alone. Results are
kept in `runs[beta] = {"model", "history"}`.

Tip on Colab: bump `TOTAL_STEPS` and add more β values once you've confirmed the
pipeline runs. Lower β → encoder ignores noise (σ→0, sharp recon, unusable
prior); higher β → KL dominates (σ→1, μ→0, blurry).

In [ ]:
BETAS             = [1e-3, 5e-3, 5e-2, 2e-1]
TOTAL_STEPS       = 3000
BETA_WARMUP_STEPS = 1000
SEED              = 0
SAVE_CKPTS        = True

runs = {}
for beta in BETAS:
    model, history = train_vae(
        beta_max=beta,
        beta_warmup_steps=BETA_WARMUP_STEPS,
        total_steps=TOTAL_STEPS,
        seed=SEED,
        val_loader=val_loader,                # held-out recon for fair comparison
    )
    runs[beta] = {"model": model, "history": history}

    final_sig = history["sigma_mean"][-1]
    final_rec = history["val_recon"][-1]
    print(f"β={beta:.0e}  →  final σ={final_sig:.3f}   val_recon={final_rec:.4f}")

    if SAVE_CKPTS:
        ckpt = os.path.join(CKPT_DIR, f"vae_beta_{beta:.0e}.pt")
        torch.save({"state_dict": model.state_dict(), "beta": beta,
                    "history": history, "total_steps": TOTAL_STEPS}, ckpt)
        print(f"   saved → {ckpt}")

In [ ]:
# ── compare curves across β ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
cmap = plt.cm.viridis(np.linspace(0, 1, len(BETAS)))

for c, beta in zip(cmap, BETAS):
    h = runs[beta]["history"]
    s = h["step"]
    axes[0].plot(s, h["val_recon"], color=c, label=f"β={beta:.0e}")
    axes[0].plot(s, h["recon"],     color=c, ls="--", alpha=0.4)
    axes[1].plot(s, h["kl"],    color=c, label=f"β={beta:.0e}")
    axes[2].plot(s, h["loss"],  color=c, label=f"β={beta:.0e}")
    axes[3].plot(s, h["sigma_mean"], color=c, label=f"β={beta:.0e}")

axes[0].set_yscale("log"); axes[0].set_title("recon MSE — val (solid) vs train (dashed)")
axes[1].set_yscale("log"); axes[1].set_title("kl (unweighted)")
axes[2].set_yscale("log"); axes[2].set_title("total loss")
axes[3].set_title("mean σ  (target 0.5–1.0)")
axes[3].axhline(1.0, ls="--", color="grey", lw=1)
axes[3].axhline(0.5, ls=":",  color="grey", lw=1)

for ax in axes:
    ax.set_xlabel("step"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.suptitle("β sweep — loss components and posterior σ")
plt.tight_layout(); plt.show()

# ── final-σ vs β: the money plot ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
final_sigs = [runs[b]["history"]["sigma_mean"][-1] for b in BETAS]
final_recs = [runs[b]["history"]["val_recon"][-1]  for b in BETAS]
ax.plot(BETAS, final_sigs, "o-", color="tab:red", label="final σ")
ax.set_xscale("log"); ax.set_xlabel("β"); ax.set_ylabel("final mean σ", color="tab:red")
ax.axhspan(0.5, 1.0, color="green", alpha=0.1, label="target band")
ax.axhline(1.0, ls="--", color="grey", lw=1)
ax2 = ax.twinx()
ax2.plot(BETAS, final_recs, "s--", color="tab:blue", label="final val recon")
ax2.set_ylabel("final val recon (MSE)", color="tab:blue")
ax.set_title("posterior collapse vs reconstruction trade-off")
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Latent-dimension sweep

Hold **β fixed** and vary `latent_channels` — the number of channels in the
spatial latent `z` (`[B, latent_channels, 32, 32]`). This probes the latent
**capacity / bottleneck**.

**The metric that matters: held-out (`val`) reconstruction.** On the *training*
set, more channels always reconstruct better — that's memorization, not a real
gain. `train_vae(..., val_loader=val_loader)` measures reconstruction on the
disjoint validation split so the comparison is fair:

- **too few channels** → bottleneck too tight, *val* recon stays high (underfit).
- **right capacity** → *val* recon bottoms out (the elbow). Pick the **smallest**
  `latent_channels` at that elbow — compact latents are easier to model downstream.
- **too many channels** → *val* recon flat or worse while *train* recon keeps
  dropping → the train/val gap widens (overfitting, unused/collapsed channels).

Same seed across runs (identical init + data order), so differences trace back
to capacity, not luck. Results land in `dim_runs[latent_channels]`.

In [ ]:
BETA_WARMUP_STEPS = 1000
TOTAL_STEPS       = 3000
LATENT_DIMS       = [2, 4, 8, 16, 32, 48]   # channels in the spatial latent z
DIM_BETA          = 2e-01               # fix β at a sensible value from the β sweep
DIM_TOTAL_STEPS   = TOTAL_STEPS
DIM_SEED          = 0
SAVE_DIM_CKPTS    = True


dim_runs = {}
for lc in LATENT_DIMS:
    model, history = train_vae(
        beta_max=DIM_BETA,
        beta_warmup_steps=BETA_WARMUP_STEPS,
        total_steps=DIM_TOTAL_STEPS,
        seed=DIM_SEED,
        desc=f"lc={lc}",
        model_kwargs={"latent_channels": lc},
        val_loader=val_loader,                # held-out recon for fair comparison
    )
    n_params = sum(p.numel() for p in model.parameters())
    dim_runs[lc] = {"model": model, "history": history, "params": n_params}

    print(f"latent_channels={lc:>2}  →  final σ={history['sigma_mean'][-1]:.3f}   "
          f"train_recon={history['recon'][-1]:.4f}   "
          f"val_recon={history['val_recon'][-1]:.4f}   params={n_params:,}")

    if SAVE_DIM_CKPTS:
        ckpt = os.path.join(CKPT_DIR, f"vae_lc_{lc}.pt")
        torch.save({"state_dict": model.state_dict(), "latent_channels": lc,
                    "beta": DIM_BETA, "history": history,
                    "total_steps": DIM_TOTAL_STEPS}, ckpt)
        print(f"   saved → {ckpt}")

In [ ]:
# ── compare curves across latent dims ──────────────────────────────────────
# axes[0] shows VAL recon (solid) vs TRAIN recon (dashed): the gap between them
# is the overfitting/memorization that makes train recon a misleading metric.
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
cmap = plt.cm.plasma(np.linspace(0, 0.9, len(LATENT_DIMS)))

for c, lc in zip(cmap, LATENT_DIMS):
    h = dim_runs[lc]["history"]
    s = h["step"]
    axes[0].plot(s, h["val_recon"], color=c, label=f"lc={lc}")
    axes[0].plot(s, h["recon"],     color=c, ls="--", alpha=0.4)
    axes[1].plot(s, h["kl"],         color=c, label=f"lc={lc}")
    axes[2].plot(s, h["loss"],       color=c, label=f"lc={lc}")
    axes[3].plot(s, h["sigma_mean"], color=c, label=f"lc={lc}")

axes[0].set_yscale("log"); axes[0].set_title("recon MSE — val (solid) vs train (dashed)")
axes[1].set_yscale("log"); axes[1].set_title("kl (unweighted)")
axes[2].set_yscale("log"); axes[2].set_title("total loss")
axes[3].set_title("mean σ  (target 0.5–1.0)")
axes[3].axhline(1.0, ls="--", color="grey", lw=1)
axes[3].axhline(0.5, ls=":",  color="grey", lw=1)

for ax in axes:
    ax.set_xlabel("step"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.suptitle(f"latent-dim sweep (β={DIM_BETA:.0e}) — loss components and posterior σ")
plt.tight_layout(); plt.show()

# ── capacity curve: held-out recon & σ vs latent_channels ──────────────────
import matplotlib.ticker as mticker

fig, ax = plt.subplots(figsize=(6, 4))
val_recs   = [dim_runs[lc]["history"]["val_recon"][-1] for lc in LATENT_DIMS]
train_recs = [dim_runs[lc]["history"]["recon"][-1]     for lc in LATENT_DIMS]
final_sigs = [dim_runs[lc]["history"]["sigma_mean"][-1] for lc in LATENT_DIMS]

ax.plot(LATENT_DIMS, val_recs,   "s-",  color="tab:blue", label="val recon (use this)")
ax.plot(LATENT_DIMS, train_recs, "v--", color="tab:cyan", alpha=0.6, label="train recon")
ax.set_xscale("log", base=2); ax.set_xticks(LATENT_DIMS)
ax.xaxis.set_major_formatter(mticker.ScalarFormatter())
ax.set_xlabel("latent_channels"); ax.set_ylabel("recon (MSE)", color="tab:blue")
ax.legend(loc="upper right", fontsize=8)

# mark the best held-out dim
best_lc = LATENT_DIMS[int(np.argmin(val_recs))]
ax.axvline(best_lc, color="green", ls=":", lw=1.5)
ax.annotate(f"best val\nlc={best_lc}", (best_lc, min(val_recs)),
            textcoords="offset points", xytext=(8, 10), color="green", fontsize=8)

ax2 = ax.twinx()
ax2.plot(LATENT_DIMS, final_sigs, "o--", color="tab:red", label="final σ")
ax2.axhspan(0.5, 1.0, color="green", alpha=0.1)
ax2.set_ylabel("final mean σ", color="tab:red")
ax.set_title("latent capacity: pick the elbow of VAL recon")
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

# ── table: train vs val recon (gap = overfitting), with elbow detection ────
print(f"{'lc':>4}  {'params':>12}  {'train_rec':>10}  {'val_rec':>10}  "
      f"{'gap':>8}  {'val Δ%':>7}  {'σ':>6}")
print("-" * 70)
prev_val = None
for lc in LATENT_DIMS:
    h   = dim_runs[lc]["history"]
    tr  = h["recon"][-1]; vl = h["val_recon"][-1]; sg = h["sigma_mean"][-1]
    gap = vl - tr
    delta = "" if prev_val is None else f"{(vl - prev_val) / prev_val * 100:+6.1f}%"
    star = "  <- best val" if lc == best_lc else ""
    print(f"{lc:>4}  {dim_runs[lc]['params']:>12,}  {tr:>10.4f}  {vl:>10.4f}  "
          f"{gap:>8.4f}  {delta:>7}  {sg:>6.3f}{star}")
    prev_val = vl
print("\nPick the smallest lc where val recon stops improving meaningfully "
      "(val Δ% near 0) — that's the capacity elbow. A widening train/val gap "
      "means you're adding channels that only memorize.")